# Projeto 1: criação das bases de treino e teste

**Ciência dos Dados, 2026.2**

Este notebook monta a base com a qual o seu grupo vai trabalhar no Projeto 1.
Rode as células na ordem. Ao final, ele salva dois arquivos CSV que você vai usar
no `Projeto1_Template.ipynb`.

O que ele faz:

1. Levanta os números da base: quantos artigos ao todo e por categoria.
2. Aplica a limpeza, com a função `limpar()` que já vem pronta aqui.
3. Mede o efeito de cada passo da limpeza.
4. Sorteia as categorias do grupo a partir dos nomes dos integrantes.
5. Balanceia por subamostragem, igualando todas ao tamanho da menor.
6. Separa em treino (70%) e teste (30%), de forma estratificada.

**Leia o enunciado do Projeto 1 antes de continuar.** A escolha de *quais*
categorias entram é feita pelo sorteio, nunca manualmente.

A base vem **sem nenhuma limpeza**, e é aqui que ela é tratada. O
`Projeto1_Template.ipynb` já recebe os dados prontos.

In [5]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Ajuste o caminho se você salvou o arquivo em outro lugar.
CAMINHO_BASE = Path("../dados/all_Article_df.parquet")

dados = pd.read_parquet(CAMINHO_BASE)
print(f"{len(dados):,} artigos")
print(f"colunas: {list(dados.columns)}")
dados.head(3)

41,919 artigos
colunas: ['Article', 'url', 'label']


,Article,url,label
0,The path to promotion for Great Britain in the...,https://www.theguardian.com/sport/2019/feb/06/...,sport
1,England have kept France guessing by naming Jo...,https://www.theguardian.com/sport/2019/feb/06/...,sport
2,Geraint Thomas is confident cycling is now “on...,https://www.theguardian.com/sport/2019/feb/06/...,sport


---
## 1. Conhecendo a base

Antes de qualquer coisa, veja com o que você está lidando. O enunciado não traz as
contagens de propósito: levantá-las é parte do trabalho.

Estes números são os da base **suja**. Depois da limpeza eles mudam, e comparar as
duas contagens já é um resultado para o relatório.

In [ ]:
print(f"Total de artigos: {len(dados):,}")
print("\nArtigos por categoria:")
print(dados["categoria"].value_counts().to_string())
print(f"\nPalavras por artigo: mediana {dados['Documento'].str.split().str.len().median():.0f}")

---
## 2. Limpeza dos dados

A base vem como veio da fonte. A função `limpar()` abaixo faz o tratamento básico
descrito no enunciado, e é o **ponto de partida**, não a resposta final.

| Passo | O que faz |
|---|---|
| L1 | remove a coluna `url`, que reproduz o rótulo |
| L2 | remove o rodapé do site no fim de cada artigo |
| L3 | remove artigos repetidos |
| L4 | normaliza aspas e travessões, que são Unicode |
| L5 | separa as palavras pela pontuação, não só pelo espaço |

Leia o código, entenda cada passo e depois vá para a seção 3, onde você vai medir
o efeito de cada um. Melhorar essa função pontua na rubrica.

In [ ]:
import re
import unicodedata

# L4: os equivalentes ASCII das aspas e travessões tipográficos
TRADUCAO = {
    0x2018: "'", 0x2019: "'", 0x201C: '"', 0x201D: '"',
    0x2013: "-", 0x2014: "-", 0x2026: " ", 0x00A0: " ",
}
# L5: uma palavra é uma sequência de letras, aceitando apóstrofo interno
PADRAO_PALAVRA = re.compile(r"[a-z]+(?:'[a-z]+)?")
# L2: o rodapé do site começa aqui
MARCA_RODAPE = re.compile(r"Explore more on these topics")


def remover_rodape(texto):
    achou = MARCA_RODAPE.search(texto)
    return texto[: achou.start()] if achou else texto


def separar_palavras(texto):
    texto = texto.translate(TRADUCAO)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return PADRAO_PALAVRA.findall(texto.lower())


def limpar(dados):
    dados = dados.copy()
    dados = dados.drop(columns=["url"])                                  # L1
    dados["Documento"] = dados["Documento"].map(remover_rodape).str.strip()  # L2
    dados = dados.drop_duplicates(subset=["Documento"])                  # L3
    return dados.reset_index(drop=True)

---
## 3. Medindo o efeito da limpeza

Aplicar a limpeza sem medir não vale nada. As células abaixo mostram o tamanho do
efeito de cada passo.

In [ ]:
# L1: a coluna url reproduz o rótulo?
secao = dados["url"].str.extract(r"theguardian\.com/([^/]+)/")[0]
print(f"L1: a url reproduz a categoria em {(secao == dados['categoria']).mean():.1%} dos artigos")
print("    por isso ela sai: um modelo que a usasse acertaria tudo sem ler o texto")

In [ ]:
# L2 e L3: o rodapé esconde repetições
print(f"L2: artigos com rodape do site: {dados['Documento'].str.contains(MARCA_RODAPE).mean():.1%}")

dup_com = dados["Documento"].duplicated().sum()
dup_sem = dados["Documento"].map(remover_rodape).str.strip().duplicated().sum()
print(f"L3: repeticoes encontradas no texto original: {dup_com:,}")
print(f"    repeticoes encontradas apos remover o rodape: {dup_sem:,}")
print(f"    a ordem importa: {dup_sem - dup_com:,} repeticoes so aparecem depois do L2")

In [ ]:
antes = len(dados)
dados = limpar(dados)
print(f"linhas antes:   {antes:,}")
print(f"linhas depois:  {len(dados):,}  ({antes - len(dados):,} removidas)")
print(f"colunas: {list(dados.columns)}")

In [ ]:
# L4 e L5: o custo de separar as palavras do jeito ingênuo
amostra = dados["Documento"].head(400)
vocab_bom = {p for t in amostra for p in separar_palavras(t)}
vocab_ruim = {p for t in amostra for p in t.lower().split()}

print(f"vocabulario em 400 artigos, separando pela pontuacao: {len(vocab_bom):,}")
print(f"vocabulario em 400 artigos, separando so por espaco:  {len(vocab_ruim):,}")
print(f"inflacao: {len(vocab_ruim) / len(vocab_bom):.1f}x")
print("\nexemplos do lixo criado ao separar so por espaco:")
print([p for p in sorted(vocab_ruim - vocab_bom) if "." in p][:8])

Anote esses números: eles entram no relatório do `Projeto1_Template.ipynb`.

**Quer pontuar?** Melhore a `limpar()`. Caminhos possíveis: truncar os artigos
muito longos nas primeiras N palavras, remover stopwords, aplicar stemming ou
lematização. Para cada mudança, justifique a escolha e registre quanto ela alterou
o número de linhas e o tamanho do vocabulário.

In [ ]:
# ESPAÇO PARA AS SUAS MELHORIAS na função limpar()

---
## 4. Dados do seu grupo

Preencha as três variáveis abaixo:

- `USERNAME`: uma combinação curta de letras que identifique o grupo. Vai no nome
  dos arquivos gerados.
- `INTEGRANTES`: o nome de cada integrante (2 para dupla, 3 para trio).
- `N_CATEGORIAS`: quantas categorias sortear.

O mínimo obrigatório é **2 para dupla** e **3 para trio**. Sortear mais do que o
mínimo (até 3 para dupla, até 5 para trio) pontua um item avançado na rubrica.

In [ ]:
USERNAME = "coloque_aqui"
INTEGRANTES = ["Nome Sobrenome", "Nome Sobrenome"]   # 2 para dupla, 3 para trio
N_CATEGORIAS = 2                                      # dupla: 2 ou 3 | trio: 3, 4 ou 5

In [ ]:
def validar(username, integrantes, n_categorias):
    if username.strip() in ("", "coloque_aqui"):
        raise ValueError("Defina um USERNAME para o seu grupo.")
    nomes = [n for n in integrantes if n.strip() and n.strip() != "Nome Sobrenome"]
    if len(nomes) != len(integrantes):
        raise ValueError("Preencha o nome de todos os integrantes.")
    if len(nomes) == 2 and n_categorias not in (2, 3):
        raise ValueError("Dupla deve sortear 2 ou 3 categorias.")
    if len(nomes) == 3 and n_categorias not in (3, 4, 5):
        raise ValueError("Trio deve sortear 3, 4 ou 5 categorias.")
    if len(nomes) not in (2, 3):
        raise ValueError("O grupo deve ter 2 ou 3 integrantes.")
    return nomes


INTEGRANTES = validar(USERNAME, INTEGRANTES, N_CATEGORIAS)
print(f"Grupo de {len(INTEGRANTES)} integrantes, sorteando {N_CATEGORIAS} categorias.")

---
## 5. O sorteio

A semente vem de um *hash* dos nomes dos integrantes. Duas propriedades
importantes:

- **É reprodutível.** Rodando de novo com os mesmos nomes, saem as mesmas
  categorias. Se você perder os arquivos, é só rodar de novo.
- **Não depende da ordem** em que os nomes foram digitados.

In [ ]:
def semente_do_grupo(nomes):
    chave = "|".join(sorted(n.strip().upper() for n in nomes))
    return int(hashlib.blake2b(chave.encode("utf-8"), digest_size=8).hexdigest(), 16)


def sortear_categorias(disponiveis, quantas, semente):
    gerador = np.random.default_rng(semente)
    return sorted(gerador.choice(sorted(disponiveis), size=quantas, replace=False).tolist())


SEMENTE = semente_do_grupo(INTEGRANTES)
CATEGORIAS = sortear_categorias(dados["categoria"].unique(), N_CATEGORIAS, SEMENTE)
RANDOM_STATE = SEMENTE % (2**32)

print(f"Integrantes: {INTEGRANTES}")
print(f"Semente:     {SEMENTE}")
print(f"Categorias:  {CATEGORIAS}")
print("\nAnote esses valores: eles são a evidência do sorteio e devem ser")
print("registrados no Projeto1_Template.ipynb.")

---
## 6. Balanceamento e separação treino/teste

As categorias da base têm tamanhos muito diferentes. Um classificador treinado
com dados desbalanceados aprende que chutar na categoria maior é uma boa
estratégia, e a acurácia global esconde esse problema.

A solução aqui é a subamostragem: todas as categorias são reduzidas ao tamanho da
menor entre as sorteadas.

In [ ]:
selecao = dados[dados["categoria"].isin(CATEGORIAS)]
n_minimo = selecao["categoria"].value_counts().min()

print("Antes do balanceamento:")
print(selecao["categoria"].value_counts().to_string())
print(f"\nMenor categoria: {n_minimo} artigos. Todas serão reduzidas a esse tamanho.")

base = (
    selecao.groupby("categoria", group_keys=False)
    .sample(n=n_minimo, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f"\nBase balanceada: {len(base)} artigos")

In [ ]:
treino, teste = train_test_split(
    base[["Documento", "categoria"]],
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=base["categoria"],
)

print("Treino:")
print(treino["categoria"].value_counts().sort_index().to_string())
print("\nTeste:")
print(teste["categoria"].value_counts().sort_index().to_string())

---
## 7. Salvando os arquivos

Estes dois arquivos são entregáveis do projeto e a entrada do
`Projeto1_Template.ipynb`.

In [ ]:
arquivo_treino = f"dados_treino_{USERNAME}.csv"
arquivo_teste = f"dados_teste_{USERNAME}.csv"

treino.to_csv(arquivo_treino, index=False)
teste.to_csv(arquivo_teste, index=False)

print(f"Salvos: {arquivo_treino} e {arquivo_teste}")
print("\nNo Projeto1_Template.ipynb, leia assim:")
print(f"    treino = pd.read_csv('{arquivo_treino}')")
print(f"    teste  = pd.read_csv('{arquivo_teste}')")